# ⚡ Advanced Hyperparameter Optimization

## Overview
This notebook focuses on optimizing hyperparameters for our best-performing models using advanced techniques including Grid Search, Random Search, and Bayesian Optimization.

### Optimization Techniques:
- 🔍 **Grid Search**: Exhaustive search over parameter grid
- 🎲 **Random Search**: Random sampling of parameter space
- 🧠 **Bayesian Optimization**: Smart parameter exploration
- 🔄 **Cross-Validation**: Robust model evaluation
- 📊 **Learning Curves**: Training progress analysis
- 🎯 **Feature Importance**: Understanding model decisions

---

In [ ]:
# Import comprehensive optimization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn optimization tools
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, cross_val_score,
    validation_curve, learning_curve, KFold
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Models for optimization
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

# Advanced optimization libraries
try:
    from skopt import BayesSearchCV
    from skopt.space import Real, Integer, Categorical
    from skopt.utils import use_named_args
    from skopt import gp_minimize
    BAYESIAN_OPT_AVAILABLE = True
    print("✅ Bayesian Optimization available")
except ImportError:
    BAYESIAN_OPT_AVAILABLE = False
    print("⚠️ Bayesian Optimization not available. Install with: pip install scikit-optimize")

try:
    import optuna
    OPTUNA_AVAILABLE = True
    print("✅ Optuna available")
except ImportError:
    OPTUNA_AVAILABLE = False
    print("⚠️ Optuna not available. Install with: pip install optuna")

# Utilities
import time
import joblib
from datetime import datetime
from IPython.display import display, HTML

print("\n⚡ Hyperparameter Optimization Setup Complete!")
print(f"🔍 Bayesian Optimization: {BAYESIAN_OPT_AVAILABLE}")
print(f"🎯 Optuna: {OPTUNA_AVAILABLE}")
print("🚀 Ready for Advanced Hyperparameter Tuning!")

In [ ]:
# Load and prepare data for optimization
print("📥 Preparing Data for Hyperparameter Optimization")
print("=" * 55)

# Load dataset
df = pd.read_csv('../ethiopia_salary_data.csv')
print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

# Prepare features and target
df['test_score'].fillna(df['test_score'].median(), inplace=True)

feature_columns = ['experience_years', 'test_score', 'department', 'education_level', 'location']
target_column = 'salary_etb'

X = df[feature_columns].copy()
y = df[target_column].copy()

# Split data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create preprocessing pipeline
numerical_features = ['experience_years', 'test_score']
categorical_features = ['department', 'education_level', 'location']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

print(f"✅ Data prepared for optimization")
print(f"📊 Training samples: {X_train.shape[0]}")
print(f"📊 Testing samples: {X_test.shape[0]}")
print(f"🎯 Target range: {y.min():,.0f} - {y.max():,.0f} ETB")

# Define evaluation function
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Comprehensive model evaluation function"""
    start_time = time.time()
    
    # Fit model
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    training_time = time.time() - start_time
    
    return {
        'model_name': model_name,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'training_time': training_time,
        'overfitting': train_r2 - test_r2
    }

print("\n🔧 Evaluation function defined")
print("⚡ Ready to start hyperparameter optimization!")

## 🔍 Grid Search Optimization

Let's start with comprehensive grid search for our top-performing models.

In [ ]:
# Grid Search Hyperparameter Optimization
print("🔍 GRID SEARCH HYPERPARAMETER OPTIMIZATION")
print("=" * 50)

# Define parameter grids for different models
param_grids = {
    'RandomForest': {
        'regressor__n_estimators': [50, 100, 200],
        'regressor__max_depth': [5, 10, 15, None],
        'regressor__min_samples_split': [2, 5, 10],
        'regressor__min_samples_leaf': [1, 2, 4],
        'regressor__max_features': ['sqrt', 'log2', None]
    },
    'GradientBoosting': {
        'regressor__n_estimators': [50, 100, 200],
        'regressor__learning_rate': [0.01, 0.1, 0.2],
        'regressor__max_depth': [3, 5, 7],
        'regressor__min_samples_split': [2, 5, 10],
        'regressor__min_samples_leaf': [1, 2, 4]
    },
    'Ridge': {
        'regressor__alpha': [0.1, 1.0, 10.0, 100.0, 1000.0],
        'regressor__solver': ['auto', 'svd', 'cholesky', 'lsqr']
    },
    'SVR': {
        'regressor__C': [0.1, 1, 10, 100],
        'regressor__gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
        'regressor__kernel': ['rbf', 'linear', 'poly']
    }
}

# Define models
models = {
    'RandomForest': RandomForestRegressor(random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'Ridge': Ridge(random_state=42),
    'SVR': SVR()
}

# Store results
grid_search_results = {}
best_models = {}

# Perform grid search for each model
for model_name, model in models.items():
    print(f"\n🔍 Optimizing {model_name}...")
    print("-" * 30)
    
    # Create pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Grid search
    grid_search = GridSearchCV(
        pipeline,
        param_grids[model_name],
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    optimization_time = time.time() - start_time
    
    # Store results
    grid_search_results[model_name] = {
        'best_score': -grid_search.best_score_,  # Convert back to positive RMSE
        'best_params': grid_search.best_params_,
        'optimization_time': optimization_time,
        'n_combinations': len(grid_search.cv_results_['params'])
    }
    
    best_models[model_name] = grid_search.best_estimator_
    
    print(f"✅ Best CV RMSE: {np.sqrt(-grid_search.best_score_):,.0f} ETB")
    print(f"⏱️  Optimization time: {optimization_time:.1f} seconds")
    print(f"🔢 Parameter combinations tested: {len(grid_search.cv_results_['params'])}")
    
    # Display best parameters
    print("🎯 Best parameters:")
    for param, value in grid_search.best_params_.items():
        print(f"   {param}: {value}")

print("\n🎉 Grid Search Optimization Complete!")